# p148 Sort List 解析笔记

- 题号：p148
- 题目英文名：Sort List
- 题目中文名：排序链表
- 题目链接：https://leetcode.cn/problems/sort-list/
- 题型：链表 / 归并排序
- 难度：Medium
- 推荐优先级：中
- Java 原代码位置：`solutions-java/leetcode/p148_sort_list/SortList.java`


## 1. 题目一句话总结

这道题要求我们在 `O(n log n)` 时间内把链表排序，并尽量满足 `O(1)` 额外空间。

本质上考的是链表上的归并排序，而 Java 原方案主推的是自底向上的迭代归并，而不是递归归并。


## 2. 题目理解与约束分析

### 2.1 题目要求

给定一个单链表头节点 `head`，把整条链表按升序排好后返回。

注意这不是数组排序，链表不能随机访问，所以很多数组上的排序技巧并不合适。

### 2.2 输入与输出

- 输入：链表头节点 `head`
- 输出：排序后的新头节点
- 返回结果含义：返回同一批节点重新连接后的升序链表

### 2.3 关键约束

- 时间复杂度要求 `O(n log n)`
- 额外空间尽量做到 `O(1)`
- 链表天然适合归并，不适合原地快速排序那种随机分区

### 2.4 示例分析

输入：`[4,2,1,3]`

排序后应得到：`[1,2,3,4]`

如果用归并排序看这件事，就是不断把小段有序链表两两合并，直到整条链表整体有序。


## 3. Java 原代码

```java
package leetcode.p148_sort_list;

public class SortList {

    public static class ListNode {
        public int val;
        public ListNode next;
    }

    public static ListNode start;
    public static ListNode end;

    public static ListNode sortList(ListNode head) {
        int n = 0;
        ListNode cur = head;
        while (cur != null) {
            n++;
            cur = cur.next;
        }

        ListNode l1;
        ListNode r1;
        ListNode l2;
        ListNode r2;
        ListNode nextTeamHead;
        ListNode lastTeamEnd;

        for (int step = 1; step < n; step <<= 1) {
            l1 = head;
            r1 = findEnd(l1, step);
            l2 = r1.next;
            r2 = findEnd(l2, step);
            nextTeamHead = r2.next;

            r1.next = null;
            r2.next = null;
            merge(l1, r1, l2, r2);

            head = start;
            lastTeamEnd = end;

            while (nextTeamHead != null) {
                l1 = nextTeamHead;
                r1 = findEnd(l1, step);
                l2 = r1.next;
                if (l2 == null) {
                    lastTeamEnd.next = l1;
                    break;
                }
                r2 = findEnd(l2, step);
                nextTeamHead = r2.next;
                r1.next = null;
                r2.next = null;
                merge(l1, r1, l2, r2);
                lastTeamEnd.next = start;
                lastTeamEnd = end;
            }
        }
        return head;
    }

    public static ListNode findEnd(ListNode s, int k) {
        while (s.next != null && --k != 0) {
            s = s.next;
        }
        return s;
    }

    public static void merge(ListNode l1, ListNode r1, ListNode l2, ListNode r2) {
        ListNode pre;
        if (l1.val <= l2.val) {
            start = l1;
            pre = l1;
            l1 = l1.next;
        } else {
            start = l2;
            pre = l2;
            l2 = l2.next;
        }

        while (l1 != null && l2 != null) {
            if (l1.val <= l2.val) {
                pre.next = l1;
                pre = l1;
                l1 = l1.next;
            } else {
                pre.next = l2;
                pre = l2;
                l2 = l2.next;
            }
        }

        if (l1 != null) {
            pre.next = l1;
            end = r1;
        } else {
            pre.next = l2;
            end = r2;
        }
    }
}
```

Java 文件里还附带了一份递归版归并排序 `sortList2`，但主方案明确强调：为了满足额外空间 `O(1)`，更推荐自底向上的迭代归并。


## 4. 先从 Java 原方案理解

Java 原方案的核心不是“归并排序”这四个字，而是它选择了自底向上版本。

也就是说：

- 先把链表看成很多长度为 `1` 的有序段
- 再两两合并成长度为 `2` 的有序段
- 再合并成长度为 `4` 的有序段
- 再合并成长度为 `8` 的有序段
- 直到整个链表有序

Java 代码里最有辨识度的几个点是：

- 用 `step` 表示每轮每个有序段的长度
- 用 `findEnd` 找到每段的尾节点
- 用全局变量 `start` 和 `end` 记录每次合并后的新头、新尾
- 每轮都要把左右两段先断开，再做 merge

这套写法比递归版更贴近题目对额外空间的要求。


## 5. 从朴素思路到优化思路

### 5.1 最直接的想法

最直观的方式可能是：

1. 把链表节点值取到数组里
2. 数组排序
3. 再写回链表或重建链表

### 5.2 为什么不够好

- 额外空间不是 `O(1)`
- 没利用链表适合归并的结构特点
- 题目本来就在引导我们用链表排序，而不是数组中转

### 5.3 优化方向

链表最适合的排序是归并排序，因为：

- 合并两个有序链表很自然
- 不需要随机访问
- 可以靠指针重连完成

进一步为了贴近 Java 原方案的 `O(1)` 额外空间目标，我们选择迭代的 bottom-up merge sort。


## 6. 核心算法讲解

### 6.1 算法名称

- 自底向上归并排序
- 链表归并排序

### 6.2 为什么想到这个算法

题目要求 `O(n log n)`，链表又不适合快速随机访问，所以归并排序几乎是最自然的选择。

如果再考虑额外空间，递归版归并还会多出调用栈开销，而 Java 原方案正是通过自底向上迭代避免了这一点。

### 6.3 关键状态

- `step`：当前每个有序段的长度
- `left`：左半段头节点
- `right`：右半段头节点
- `cur`：当前还没处理的链表起点
- `prev`：已经合并完成部分的尾节点

### 6.4 步骤拆解

1. 先统计链表长度 `n`
2. 令 `step = 1`，表示先合并长度为 1 的小段
3. 每轮遍历链表，把链表切成很多 `left` 和 `right` 两段
4. 把这两段有序链表合并
5. 接回总链表，并更新尾指针
6. 令 `step *= 2`，继续合并更长的有序段
7. 直到 `step >= n`


## 7. 过程演示

以链表 `[4,2,1,3]` 为例：

### 7.1 `step = 1`

每段长度先看成 1：

```text
[4] [2] [1] [3]
```

两两合并后：

```text
[2,4] [1,3]
```

### 7.2 `step = 2`

现在每段长度为 2：

```text
[2,4] [1,3]
```

再合并后：

```text
[1,2,3,4]
```

这时整条链表已经有序。


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional


@dataclass
class ListNode:
    val: int = 0
    next: Optional["ListNode"] = None


def build_linked_list(values: list[int]) -> Optional[ListNode]:
    dummy = ListNode()
    cur = dummy
    for value in values:
        cur.next = ListNode(value)
        cur = cur.next
    return dummy.next


def linked_list_to_list(head: Optional[ListNode]) -> list[int]:
    ans: list[int] = []
    while head is not None:
        ans.append(head.val)
        head = head.next
    return ans


In [ ]:
class Solution:
    def sortList(self, head: Optional[ListNode]) -> Optional[ListNode]:
        if head is None or head.next is None:
            return head

        n = 0
        cur = head
        while cur is not None:
            n += 1
            cur = cur.next

        dummy = ListNode(0, head)
        step = 1

        while step < n:
            prev = dummy
            cur = dummy.next

            while cur is not None:
                left = cur
                right = self._split(left, step)
                cur = self._split(right, step)

                merged_head, merged_tail = self._merge(left, right)
                prev.next = merged_head
                prev = merged_tail

            step <<= 1

        return dummy.next

    def _split(self, head: Optional[ListNode], step: int) -> Optional[ListNode]:
        if head is None:
            return None

        for _ in range(step - 1):
            if head.next is None:
                break
            head = head.next

        second = head.next
        head.next = None
        return second

    def _merge(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> tuple[Optional[ListNode], Optional[ListNode]]:
        dummy = ListNode()
        tail = dummy

        while l1 is not None and l2 is not None:
            if l1.val <= l2.val:
                tail.next = l1
                l1 = l1.next
            else:
                tail.next = l2
                l2 = l2.next
            tail = tail.next

        tail.next = l1 if l1 is not None else l2

        while tail.next is not None:
            tail = tail.next

        return dummy.next, tail


## 8. 代码逐段讲解

### 8.1 为什么 Python 没完全照抄 Java 变量形式

Java 原解里用全局变量 `start` 和 `end` 传出每次 merge 的结果头尾。Python 这里没有生硬照抄全局变量，而是让 `_merge` 直接返回 `(merged_head, merged_tail)`。

但核心方案并没有变，仍然是同一套自底向上归并逻辑。

### 8.2 `_split` 做什么

`_split(head, step)` 的作用相当于 Java 原解里的 `findEnd + 断链`。

它会：

- 从当前头节点出发走 `step` 个位置
- 把这一段截断
- 返回下一段的起点

### 8.3 `_merge` 做什么

`_merge` 对应 Java 原解里的 `merge(l1, r1, l2, r2)`。

它把两个有序链表合并成一个新的有序链表，并返回：

- 合并后头节点
- 合并后尾节点

### 8.4 主循环怎么推进

外层循环控制 `step = 1, 2, 4, 8...`，内层循环负责在当前 `step` 下把整条链表切成很多对小段，逐对合并。

每做完一轮，链表中有序段的长度就翻倍。

### 8.5 Java 文件里的递归版

Java 文件还提供了递归归并版本 `sortList2`，它逻辑也正确，而且更优雅。但原文件明确指出：递归版会引入调用栈开销，不完全满足额外空间 `O(1)` 的目标，所以主方案仍然是当前这个迭代版。


## 9. 复杂度分析

- 时间复杂度：`O(n log n)`
- 为什么是这个时间复杂度：每一轮会线性扫完整条链表，而 `step` 会翻倍，轮数是 `log n`
- 空间复杂度：`O(1)`
- 为什么是这个空间复杂度：除了少量指针变量外，没有额外与输入规模成比例的辅助结构


## 10. 易错点

1. 没有在切段时把链表断开，后面 merge 时就会串链甚至成环。
2. `_split` 步数控制写错，导致每段长度不是 `step`。
3. 合并完一段后没有正确维护 `prev` 指针，结果后续链段接丢。
4. 只会写递归版，却忽略了 Java 原方案强调的是迭代版空间优势。
5. 合并后没拿到尾节点，导致下一段无法正确接回总链表。


## 11. 面试时怎么讲

可以这样概括：

> 这题我会用链表归并排序做，因为题目要求 `O(n log n)`，而链表又特别适合合并有序段。
> 如果只追求时间复杂度，递归归并就够了；但 Java 原方案更进一步，用了自底向上的迭代归并，避免递归栈开销。
> 我会让 `step` 从 1 开始不断翻倍，每轮把链表拆成很多对长度为 `step` 的有序段，然后逐对合并。
> 最终时间复杂度 `O(n log n)`，额外空间复杂度 `O(1)`。


## 12. 实际应用场景

这题可以对应到：对链式或流式组织的数据做稳定排序，而不能先整体搬进数组。

### 具体业务案例：日志链式缓冲区排序

#### 业务背景

某些系统会把日志片段或任务片段组织成链式结构，而不是连续数组。

#### 输入是什么

输入是一条链表，每个节点表示一条日志记录或任务记录，包含待排序字段。

#### 算法介入点

系统需要按时间戳或优先级重新排序，但又不想额外复制整批数据到大数组中。

#### 输出是什么

输出是一条按目标字段有序的新链表结构。

#### 价值是什么

它解决的是链式数据在低额外空间条件下的高效排序问题。


In [ ]:
head = build_linked_list([4, 2, 1, 3])
print('示例一:', linked_list_to_list(Solution().sortList(head)))

head = build_linked_list([-1, 5, 3, 4, 0])
print('示例二:', linked_list_to_list(Solution().sortList(head)))

head = build_linked_list([1])
print('单节点:', linked_list_to_list(Solution().sortList(head)))


## 13. Demo 输出说明

- 第一组输出应为 `[1, 2, 3, 4]`，说明基础排序正确。
- 第二组输出应为 `[-1, 0, 3, 4, 5]`，说明对包含负数和乱序数据也能正常排序。
- 第三组输出应为 `[1]`，说明边界情况下不会破坏链表结构。


## 14. 一句话复盘

> 这题最关键的不是记住归并排序，而是理解 Java 原方案为什么坚持用自底向上的迭代归并来贴近 `O(1)` 额外空间目标。
